<a href="https://colab.research.google.com/github/cpdong/public/blob/master/test/demo_TPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🧬 LLMBind - A sequence-to-sequence protein binder designer

**De novo protein binder design from a target.**

An LLM generates candidate binder sequences for a target, which are then filtered by a PPI
classifier (*bindscan*) and, optionally, a structure filter (*NetSurfP-3.0*).

**Runtime:** this is a PyTorch/CUDA pipeline — set **Runtime → Change runtime type → GPU (T4)**.
_(No TPU / JAX / TensorFlow is used.)_

**Two steps:**
1. **Install** (~4 min, run once) — also fetches the demo target (`PDL1.fasta`) and pre-downloads
   the generation model from HuggingFace (`cpdong/test`).
2. **Generate binders** — uses the bundled `PDL1.fasta` target by default. Nothing is fetched
   from GitHub or HuggingFace at run time.


In [ ]:
#@title 1. Install LLMBind  &  fetch demo + model (~4 min){ display-mode: "form" }
#@markdown Run this **once**. Installs the PyTorch-based pipeline
#@markdown (`transformers` + `fair-esm` + NetSurfP-3.0), fetches the demo target
#@markdown (`PDL1.fasta`), and pre-downloads the generation model from HuggingFace.
#@markdown No TPU / JAX / TensorFlow needed — uses Colab's built-in CUDA PyTorch.

#@markdown ---
#@markdown Generation model on the Hub. Only needed here if it is **private/gated** —
#@markdown leave blank for public repos.
hf_model_repo = "cpdong/test"  #@param {type:"string"}
hf_token      = ""             #@param {type:"string"}
#@markdown Tick to re-download the model even if it's already on disk.
force_download = False         #@param {type:"boolean"}

import os, time, subprocess

t0 = time.time()
WORK_DIR = "/content/LLMBind_demo"
LLM_DIR  = os.path.join(WORK_DIR, "llm_model")
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

def sh(cmd):
    print(">>", cmd)
    if subprocess.run(cmd, shell=True).returncode != 0:
        raise RuntimeError("command failed: " + cmd)

if not os.path.isfile(os.path.join(WORK_DIR, "READY")):
    # Python deps. torch / torchvision ship with Colab (CUDA build) — don't reinstall them.
    sh("pip -q install "
       "transformers==4.57.3 accelerate fair-esm 'huggingface_hub>=0.23' "
       "'numpy==1.26.4' scipy==1.13.1 scikit-learn==1.6.1 "
       "pandas h5py 'pyyaml==6.0.2' 'requests==2.32.3' biopython matplotlib")

    # NetSurfP-3.0 (structure filter) — a torch-based fork, no TensorFlow
    sh("pip -q install git+https://github.com/cpdong/NetSurfP_3.0_standalone.git")

    # Preload ESM weights into the torch hub cache
    sh('python -c "import esm; esm.pretrained.esm1b_t33_650M_UR50S()"')  # NetSurfP-3.0
    sh('python -c "import esm; esm.pretrained.esm2_t6_8M_UR50D()"')      # bindscan PPI

    open(os.path.join(WORK_DIR, "READY"), "w").close()
    print("\n✅ Dependencies ready.")
else:
    print("Dependencies already installed (delete /content/LLMBind_demo/READY to reinstall).")

# --- Demo + PPI files: fetched if missing (independent of the install gate). ---
BASE = "https://raw.githubusercontent.com/cpdong/public/refs/heads/master/test"
for f in ["run_test.py", "bs_model.pt", "PDL1.fasta"]:
    if not os.path.isfile(os.path.join(WORK_DIR, f)):
        sh(f"wget -q {BASE}/{f} -O {f}")
print("Demo files present:", [f for f in ["run_test.py", "bs_model.pt", "PDL1.fasta"]])

# --- Ensure torchvision matches torch (independent of the install gate). ---
# NetSurfP-3.0 does `import torchvision` at module load. If torch/torchvision
# versions drift (a dep can bump torch), torchvision's C++ ops fail to register
# -> "operator torchvision::nms does not exist", which crashes run_test.py even
# with --no_structure_filter. Test in a fresh subprocess, repair if broken.
def _tv_works():
    return subprocess.run(
        ["python", "-c", "import torchvision; from torchvision.ops import nms"],
        capture_output=True, text=True,
    ).returncode == 0

if _tv_works():
    print("torchvision OK (matches torch).")
else:
    import torch
    tver = torch.__version__.split("+")[0]
    parts = tver.split(".")
    # torchvision's minor version tracks torch's minor + 15
    # (torch 2.9 -> tv 0.24, 2.10 -> 0.25, 2.11 -> 0.26, 2.12 -> 0.27, ...).
    try:
        tv_minor = int(parts[1]) + 15
        TV = f"0.{tv_minor}.*"
    except (IndexError, ValueError):
        TV = None
    print(f"⚠️  torchvision mismatch for torch {tver}; reinstalling torchvision {TV or '?'} ...")
    if TV:
        sh(f"pip -q install --no-deps --force-reinstall 'torchvision=={TV}'")
        print("✅ torchvision realigned." if _tv_works()
              else "⚠️  Still failing — do Runtime → Restart session, then re-run step 1.")
    else:
        print(f"⚠️  Could not parse torch version {tver}. Tell me this version and I'll fix it.")

# --- Generation model: downloaded independently, so it runs even if deps are
#     already installed and even if a previous run never fetched it. ---
def _weight_files(d):
    if not os.path.isdir(d):
        return []
    return [f for f in os.listdir(d)
            if f.endswith((".safetensors", ".bin", ".gguf", ".pt"))]

existing = _weight_files(LLM_DIR)
print(f"\nModel dir: {LLM_DIR}")
print(f"Weight files already there: {existing if existing else 'none'}")

if existing and not force_download:
    print("→ Skipping download (already present). Tick `force_download` above to re-pull.")
else:
    reason = "force_download=True" if existing else "no weights found"
    print(f"→ Downloading '{hf_model_repo}'  ({reason}) ...")
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"   # make progress visible
    from huggingface_hub import snapshot_download
    try:
        local_path = snapshot_download(
            repo_id=hf_model_repo,
            local_dir=LLM_DIR,
            token=(hf_token or None),
            force_download=force_download,
        )
        got = _weight_files(LLM_DIR)
        if not got:
            raise RuntimeError(
                f"Download finished but no weight file in {LLM_DIR}. "
                f"All files: {os.listdir(LLM_DIR)}"
            )
        print(f"✅ Model downloaded to {local_path}")
        print("   Files:", sorted(os.listdir(LLM_DIR)))
    except Exception as e:
        print(
            f"\n⚠️  Could not download '{hf_model_repo}': {e!r}\n"
            "    If the repo is private/gated, paste an access token in `hf_token` above\n"
            "    (https://huggingface.co/settings/tokens) and re-run this cell."
        )
        raise

# numpy was pinned to <2; if it was downgraded after import you may need to
# Runtime → Restart session once, then continue at step 2 (no need to reinstall).
print(f"\nElapsed: {time.time() - t0:.0f}s")


In [ ]:
#@title 2. Generate binders{ display-mode: "form" }

#@markdown Generates binders for the target FASTA using the model downloaded in
#@markdown step 1, then shows the results as a table you can download.

#@markdown ### Target
#@markdown Defaults to the bundled **PDL1.fasta**. To use your own target, upload a
#@markdown FASTA via the 📁 Files panel (or `files.upload()`) into `/content/LLMBind_demo/`
#@markdown and point this at it, e.g. `/content/LLMBind_demo/other_protein.fasta`.
target_fasta = "/content/LLMBind_demo/PDL1.fasta"  #@param {type:"string"}

#@markdown ### Design options
num_designs         = 10   #@param {type:"integer"}
min_length          = 70   #@param {type:"integer"}
max_length          = 130  #@param {type:"integer"}
generate_batch_size = 16   #@param {type:"integer"}

#@markdown ### PPI (bindscan) filter
#@markdown The optimal threshold for the PPI interaction prediction model varies
#@markdown across targets and should **not** be universally set to 0.5. For example,
#@markdown ~0.5 may suit **PD-L1**, whereas **PD-1** may need a substantially lower
#@markdown threshold (potentially **< 0.1**).
#@markdown <br>💡 If generation feels slow, **lower** the threshold — more candidates
#@markdown pass the filter, so fewer generation rounds are needed to reach `num_designs`.
ppi_threshold = 0.3  #@param {type:"number"}

#@markdown ### Structure filter (NetSurfP-3.0) — optional
#@markdown Turn on and give an `nsp3.pth` weights file to enable it.
enable_structure_filter = False  #@param {type:"boolean"}
nsp3_model              = ""     #@param {type:"string"}
min_structured_fraction = 0.6    #@param {type:"number"}

# ---------------------------------------------------------------------------
import os, shlex, subprocess
from pathlib import Path

WORK_DIR     = "/content/LLMBind_demo"
llm_model    = os.path.join(WORK_DIR, "llm_model")      # default download location (step 1)
output_dir   = "/content/llmbind_output"                # internal; results shown below

if not os.path.isfile(target_fasta):
    raise FileNotFoundError(f"Target FASTA not found: {target_fasta}. Run step 1 first.")
if not os.path.isdir(llm_model) or not os.listdir(llm_model):
    raise FileNotFoundError(f"Generation model not found at {llm_model}. Run step 1 first.")

Path(output_dir).mkdir(parents=True, exist_ok=True)

cmd = [
    "python", os.path.join(WORK_DIR, "run_test.py"),
    "--target_fasta",        target_fasta,
    "--gen_model",           llm_model,
    "--ppi_model",           os.path.join(WORK_DIR, "bs_model.pt"),
    "--ppi_threshold",       str(ppi_threshold),
    "--num_designs",         str(num_designs),
    "--generate_batch_size", str(generate_batch_size),
    "--min_length",          str(min_length),
    "--max_length",          str(max_length),
    "--output_dir",          output_dir,
]
if enable_structure_filter and nsp3_model:
    cmd += ["--nsp3_model", nsp3_model, "--min_structured_fraction", str(min_structured_fraction)]
else:
    cmd += ["--no_structure_filter"]

print("Running:\n  " + " ".join(shlex.quote(x) for x in cmd) + "\n", flush=True)

# Stream the child's stdout+stderr live so any error is visible inline.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(
        f"run_test.py exited with code {proc.returncode}. The actual error is in the log above."
    )

# ---- Show results as a table + offer a download -------------------------
# Pure-Python (stdlib csv) rendering — avoids importing pandas/numpy, which
# clash here because numpy is pinned to 1.26.4 for the pipeline.
import csv, html
from IPython.display import display, HTML

results_tsv = os.path.join(output_dir, "llmbind_designed_binders.tsv")
if not os.path.isfile(results_tsv):
    print("\nNo results file was produced — check the log above.")
else:
    with open(results_tsv) as f:
        rows = list(csv.DictReader(f, delimiter="\t"))

    def _num(x):
        try:
            return float(x)
        except (TypeError, ValueError):
            return float("-inf")

    if rows and "PPI_prob" in rows[0]:
        rows.sort(key=lambda r: _num(r.get("PPI_prob")), reverse=True)

    # show every column except the long, identical target_seq
    cols = [c for c in rows[0].keys() if c != "target_seq"] if rows else []
    seq_cols = {"binder_seq", "binder_optimize_seq"}

    def _cell(col, val):
        v = val if val is not None else ""
        if col in seq_cols and len(v) > 48:        # truncate long sequences on screen
            v = v[:48] + "…"
        return html.escape(v)

    head = "".join(f"<th style='padding:4px 10px;text-align:left'>{html.escape(c)}</th>" for c in cols)
    body = "".join(
        "<tr>" + "".join(
            f"<td style='padding:4px 10px;font-family:monospace;font-size:12px;white-space:nowrap'>{_cell(c, r.get(c))}</td>"
            for c in cols
        ) + "</tr>"
        for r in rows
    )
    table = (
        "<div style='overflow-x:auto;max-height:480px'>"
        "<table style='border-collapse:collapse;font-size:12px'>"
        f"<thead style='position:sticky;top:0;background:#eee'><tr>{head}</tr></thead>"
        f"<tbody>{body}</tbody></table></div>"
    )
    print(f"\n✅ {len(rows)} binders designed (sorted by predicted PPI probability):")
    display(HTML(table))

    # write a CSV copy (full, untruncated sequences) and offer an opt-in
    # download *button*. It renders inert — the file is only sent on click.
    results_csv = os.path.join(output_dir, "llmbind_designed_binders.csv")
    with open(results_tsv) as fin, open(results_csv, "w", newline="") as fout:
        w = csv.writer(fout)
        for row in csv.reader(fin, delimiter="\t"):
            w.writerow(row)

    def _download_button(path):
        try:
            # Real Colab button: nothing happens until the user clicks it.
            import ipywidgets as widgets
            from google.colab import files
            btn = widgets.Button(description="Download results (CSV)", icon="download",
                                 button_style="primary",
                                 layout=widgets.Layout(width="230px", margin="10px 0"))
            out = widgets.Output()
            def _on_click(_):
                with out:
                    files.download(path)
            btn.on_click(_on_click)
            display(btn, out)
        except Exception:
            # Fallback for non-Colab environments: a click-to-download link.
            import base64
            b64 = base64.b64encode(open(path, "rb").read()).decode()
            display(HTML(
                f'<a download="llmbind_designed_binders.csv" '
                f'href="data:text/csv;base64,{b64}" '
                'style="display:inline-block;margin-top:10px;padding:8px 16px;'
                'background:#1a73e8;color:#fff;border-radius:6px;text-decoration:none;'
                'font-family:sans-serif;font-size:13px;font-weight:600">'
                '⬇️  Download results (CSV)</a>'))

    _download_button(results_csv)
    print(f"(also saved to {results_csv} — available in the 📁 Files panel)")

    # ---- Suggested next step: validate the designs' structures ----------
    display(HTML(
        "<div style='margin-top:16px;padding:12px 14px;border-left:4px solid #1a73e8;"
        "background:#f5f8ff;font-family:sans-serif;font-size:13px;line-height:1.5'>"
        "<b>🔬 Suggested next step — validate the structures.</b><br>"
        "These are sequence-level designs. To check whether a binder actually folds and "
        "engages the target, predict its structure (or the binder–target complex) with a "
        "structure model, e.g.:"
        "<ul style='margin:8px 0 0 0;padding-left:20px'>"
        "<li><a href='https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb' "
        "target='_blank'>ColabFold (AlphaFold2)</a> — fold the binder, or co-fold binder + target "
        "(AlphaFold2-multimer) and inspect interface confidence (pLDDT / ipTM).</li>"
        "<li><a href='https://esmatlas.com/resources?action=fold' target='_blank'>ESMFold</a> — "
        "fast single-sequence folding to screen binder foldability.</li>"
        "</ul></div>"))
